# Step7 — functional connectivity across independently configured variants

This notebook uses the current `step6b_fc_geometry_coverage.py` API (without a version suffix). Step6b determines eligible recordings, complete selection groups and ROI-level PASS conditions. Step7 does **not** select replicate recordings using `variant_keys`.

**Method:** select a nominal 60-second window; resample to the FC grid with finite-mask-aware area weighting; subtract the configured trend (linear by default); apply a fourth-order Butterworth SOS filter (band-pass 0.08–0.40 Hz or low-pass below 0.40 Hz, as specified in the selected variant); trim 2 seconds at each boundary by default; compute pixelwise Pearson correlations; and aggregate Fisher-transformed correlations (`z = arctanh(r)`). Results for different `rest_variant` and `variant_id` values are stored in separate directories.

The four configured signal/filter combinations are possible analytical variants, not four automatic replicates. **The code cell below currently sets `VARIANTS = ["hbt_lowpass"]`, so only that one variant runs unless you explicitly change the list.** Figures are descriptive only; no inferential tests or scientific conclusions are produced here.


## 1. Paths, REST branch, and selected signal/filter variants

Specify the intended configuration path, REST branch and variants before running any computation. The code cell contains local absolute paths that are intentionally left unchanged in this documentation-only revision. It currently sets `RUN_BATCH = True` and `OVERWRITE = True`; running the batch cell will recompute and overwrite saved products of the selected variant. For read-only inspection, set `RUN_BATCH = False` yourself before execution.


In [ ]:
from pathlib import Path
import sys, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_CODE_ROOT = Path(r"...\WIFIOPIA_PT2021")
SRC_DIR = PROJECT_CODE_ROOT / "src"
PROJECT_CONFIG = Path(r"config.yaml")

# Use ONLY the Step6b branch that has already been computed.
REST_VARIANT = "with_rest"  # "with_rest" | "without_rest"
VARIANTS = ["hbt_lowpass"] # "hbt_bandpass", , "hbt_vreg_bandpass", "hbt_vreg_lowpass"
MASK_SETS = ["stroke_longitudinal", "prestroke_stability", "daypt_anesthesia"]
ONLY_MICE = None   # Example: ["m10"] writes results to a separate _subset_runs directory.
ONLY_STEMS = None
OVERWRITE = True
RUN_BATCH = True
RUN_SUMMARIES = False  # One batch run automatically generates the mouse-level summary.

if not SRC_DIR.is_dir() and (Path.cwd() / "src" / "step7_functional_connectivity.py").exists():
    SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
import step7_functional_connectivity as s7
importlib.reload(s7)
print("Step7:", s7.__file__)
print("Step6b:", s7.s6b.__file__)
print("REST branch:", REST_VARIANT)


## 2. Check the upstream Step6b selection (without reading Zarr or computing maps)

Only complete Step6b selection groups are eligible. The completeness rule is evaluated on the full original Step6b selection **before** optional mouse/stem filtering. This cell displays selected stems and group counts; it does not recompute connectivity.


In [ ]:
# The Step6b mask-set selection must correspond to complete selection groups;
# Step7 excludes incomplete mouse-by-mask-set selections BEFORE mouse filtering.
idx, cfg, project_paths, s6b_paths, tables = s7.build_step7_index(
    PROJECT_CONFIG, mask_sets=MASK_SETS, mice=ONLY_MICE, stems=ONLY_STEMS,
    runtime_overrides={"rest_variant": REST_VARIANT},
)
print("Источник Step6b:", s6b_paths["root"])
print("Всего выбранных stem × mask_set:", len(idx))
if not idx.empty:
    display(idx.groupby(["mask_set", "comparison_group"], dropna=False).agg(
        n_stems=("stem", "nunique"), n_mice=("mouse_id", "nunique")
    ).reset_index())
    display(idx[[c for c in ["mask_set", "slot", "mouse_id", "experimental_group", "comparison_group", "state", "stem"] if c in idx.columns]].head(15))


## 3. Compute each selected variant in its own directory (optional)

If `RUN_BATCH` is enabled, `run_step7_variants` processes the variants in `VARIANTS`, with the selected REST branch and mask sets. With `OVERWRITE = True`, existing computed outputs for a stem may be deleted before new results are written. Use `RUN_BATCH = False` to skip batch processing.


In [ ]:
if RUN_BATCH:
    result_by_variant = s7.run_step7_variants(
        PROJECT_CONFIG, rest_variant=REST_VARIANT, variants=VARIANTS,
        mask_sets=MASK_SETS, mice=ONLY_MICE, stems=ONLY_STEMS,
        overwrite=OVERWRITE, save_qc=True,
        save_roi_traces=(ONLY_MICE is not None and len(ONLY_MICE) == 1),
    )
    for variant, result in result_by_variant.items():
        print(variant, '→', result['paths']['root'], 'errors:', len(result['errors']))
else:
    print('RUN_BATCH=False — существующие результаты не изменены')


## 4. Load stored results without recomputing connectivity

This section loads saved indices, ROI tables, whole-cortex mouse summaries and windows from variant-specific directories. Although the section can compare four variants, it loads **only the currently selected `VARIANTS`**. Missing results are reported and skipped.


In [ ]:
# Tables and maps are read from the same REST branch and mask set.
from pandas.errors import EmptyDataError

runs = {}
for variant in VARIANTS:
    vcfg = s7.variant_config(cfg, variant, REST_VARIANT)
    if ONLY_MICE or ONLY_STEMS:
        import json, hashlib
        subset = {"mice": sorted(str(x) for x in (ONLY_MICE or [])),
                  "stems": sorted(str(x) for x in (ONLY_STEMS or [])),
                  "mask_sets": sorted(str(x) for x in (MASK_SETS or []))}
        vcfg['functional_connectivity']['_subset_output_key'] = hashlib.sha256(json.dumps(subset, sort_keys=True).encode()).hexdigest()[:12]
    paths7 = s7.build_step7_paths(vcfg, project_paths)
    if not paths7['step7_index_csv'].exists():
        print('Нет результатов:', variant, paths7['root']); continue
    rows = pd.read_csv(paths7['step7_index_csv'])
    def load_table(path):
        try:
            return pd.read_csv(path) if path.exists() else pd.DataFrame()
        except EmptyDataError:
            return pd.DataFrame()
    runs[variant] = {
        'paths': paths7, 'index': rows,
        'homotopic': load_table(paths7['homotopic_roi_csv']),
        'global': load_table(paths7['global_roi_csv']),
        'mouse': load_table(paths7['tables_root'] / 'whole_cortex_fc_mouse_values.csv'),
        'windows': load_table(paths7['windows_csv']),
        'errors': load_table(paths7['errors_csv']),
    }
print('Доступные варианты:', list(runs))
display(pd.DataFrame([{'variant': v, 'stems': len(d['index']), 'mouse_values': len(d['mouse']),
                       'windows': len(d['windows']), 'errors': len(d['errors'])} for v,d in runs.items()]))


## 5. Rebuild mouse-level summaries from frozen maps, if required

`RUN_SUMMARIES = True` calls the summary-only addon for the currently selected variants. It uses previously saved Step7 geometry and maps and does not recalculate correlations from signal Zarr. The saved stem and mouse summary CSV files can nevertheless be updated or overwritten.


In [ ]:
if RUN_SUMMARIES:
    for variant in VARIANTS:
        vcfg = s7.variant_config(cfg, variant, REST_VARIANT)
        if ONLY_MICE or ONLY_STEMS:
            vcfg['functional_connectivity']['_subset_output_key'] = hashlib.sha256(json.dumps(subset, sort_keys=True).encode()).hexdigest()[:12]
        out = s7.run_whole_cortex_summary_from_existing_step7(
            PROJECT_CONFIG, mask_sets=MASK_SETS, mice=ONLY_MICE, stems=ONLY_STEMS,
            runtime_overrides=vcfg['functional_connectivity'],
        )
        print(variant, 'mouse rows:', len(out['mouse_values']), 'errors:', len(out['errors']))
else:
    print('Сводки не перезаписаны; RUN_SUMMARIES=False')


## 6. Descriptive comparison at the individual-mouse level

Use the stored, QC-passing mouse-level Fisher-z values for visualization. Each mouse is represented as an individual observation. For each variant, the plotted descriptive mean is calculated in Fisher-z space and then mapped back using `tanh(mean_z)`; this is **not** an arithmetic mean of Pearson-r values. No statistical hypothesis tests are performed.


In [ ]:
# Plot configuration; no statistical tests are performed.
PLOT_MASK_SET = "daypt_anesthesia"
PLOT_METRIC_KIND = "homotopic"   # homotopic | global
PLOT_HEMISPHERE = "homotopic"     # Options: homotopic | left | right | bilateral | healthy
PLOT_ROI_METRIC = None           # None displays all available metrics.
PLOT_SLOT = None                 # None includes all slots in the chosen mask_set.
PLOT_YLIM = (-1, 1)

mouse_parts = []
for vid, d in runs.items():
    mv = d['mouse']
    if mv.empty: continue
    mv = mv.copy()
    mv['variant_id'] = vid
    mouse_parts.append(mv)
all_mouse = pd.concat(mouse_parts, ignore_index=True) if mouse_parts else pd.DataFrame()
if all_mouse.empty:
    print('Нет mouse-level таблиц. Сначала рассчитайте Step7 и сводки.')
else:
    z = all_mouse[all_mouse['mask_set'].eq(PLOT_MASK_SET) & all_mouse['fc_kind'].eq(PLOT_METRIC_KIND)].copy()
    if PLOT_METRIC_KIND == 'homotopic':
        z = z[z['hemisphere_scope'].eq('homotopic')]
    else:
        z = z[z['hemisphere_scope'].eq(PLOT_HEMISPHERE)]
    if PLOT_ROI_METRIC: z = z[z['metric'].eq(PLOT_ROI_METRIC)]
    if PLOT_SLOT: z = z[z['slot'].eq(PLOT_SLOT)]
    z['value_z'] = pd.to_numeric(z['value_z'],errors='coerce')
    z = z[np.isfinite(z['value_z']) & z['analysis_pass_publication'].astype(str).str.lower().isin(['true','1'])]
    if z.empty:
        print('Нет допустимых mouse-level значений при выбранных параметрах.')
    else:
        group_col = 'comparison_group' if PLOT_MASK_SET == 'daypt_anesthesia' else 'experimental_group'
        fig, ax = plt.subplots(figsize=(max(9,len(VARIANTS)*2.3),5))
        pos = {v:i for i,v in enumerate(VARIANTS)}
        colors = plt.get_cmap('tab20')
        for mouse_idx, (mouse,g) in enumerate(z.groupby('mouse_id',sort=True)):
            for _,r in g.iterrows():
                ax.scatter(pos[r['variant_id']]+((mouse_idx%7)-3)*.024, np.tanh(float(r['value_z'])),
                           color=colors(mouse_idx%20),s=32,alpha=.8,zorder=3)
        # Give each MOUSE equal weight: average Fisher z before applying tanh.
        for v,g in z.groupby('variant_id'):
            mouse_z = g.groupby('mouse_id')['value_z'].mean()
            ax.scatter(pos[v],np.tanh(mouse_z.mean()),marker='_',s=600,linewidths=3,c='black',zorder=7)
        ax.set_xticks(range(len(VARIANTS)),VARIANTS,rotation=15)
        ax.set_ylim(PLOT_YLIM);ax.set_ylabel('Pearson r (tanh of mean Fisher z)')
        ax.set_title(f'{PLOT_MASK_SET} • {PLOT_METRIC_KIND} • individual mice and descriptive mean')
        ax.grid(axis='y',alpha=.15);plt.show()
        display(z[['variant_id','mouse_id',group_col,'slot','metric','hemisphere_scope','value_z','value_r']].head(20))


## 7. ROI correlations by variant, keeping individual mice separate

Load homotopic or GBC ROI summaries for the chosen mask set and comparison group. Within each mouse, stem repeats sharing the same slot and ROI are averaged in Fisher-z space before inverse transformation. The four panels correspond to the ROI acronyms specified in `ROI_NAMES`. The visualization uses only QC-passing rows and is descriptive.


In [ ]:
ROI_MASK_SET = 'daypt_anesthesia'
ROI_COMPARISON_GROUP = 'Iso'       # 'Awake' | 'Iso' for daypt_anesthesia
ROI_KIND = 'homotopic'            # 'homotopic' | 'global'
ROI_HEMISPHERE = 'left'           # For global: left | right
ROI_NAMES = ['MOp','MOs','SS','RSP']
ROI_YLIM = (-1, 1)

roi_parts=[]
for vid, d in runs.items():
    src=d['homotopic' if ROI_KIND=='homotopic' else 'global']
    if src.empty: continue
    zz=src[src['mask_set'].astype(str).eq(ROI_MASK_SET)].copy()
    if ROI_MASK_SET=='daypt_anesthesia':
        zz=zz[zz['comparison_group'].astype(str).eq(ROI_COMPARISON_GROUP)]
    if ROI_KIND=='global':
        zz=zz[zz['hemisphere'].astype(str).eq(ROI_HEMISPHERE)]
    zcol='mean_z' if ROI_KIND=='homotopic' else 'mean_GBC_z'
    zz['z']=pd.to_numeric(zz[zcol],errors='coerce')
    zz=zz[np.isfinite(zz['z']) & zz['analysis_pass'].astype(str).str.lower().isin(['true','1'])]
    zz['variant_id']=vid
    if not zz.empty: roi_parts.append(zz[['mouse_id','slot','analysis_roi_acronym','variant_id','z']])
if not roi_parts:
    print('Для выбранной ROI-комбинации нет данных (или Step6b PASS не пройден).')
else:
    roi_data=pd.concat(roi_parts,ignore_index=True)
    # Average stem repeats within each mouse/slot in Fisher-z space, then apply tanh.
    roi_data=roi_data.groupby(['mouse_id','slot','analysis_roi_acronym','variant_id'],as_index=False)['z'].mean()
    roi_data['r']=np.tanh(roi_data['z'])
    fig,axes=plt.subplots(2,2,figsize=(12,8),sharey=True,layout='constrained')
    for ax,roi in zip(axes.flat,ROI_NAMES):
        sub=roi_data[roi_data['analysis_roi_acronym'].astype(str).eq(roi)]
        for mouse,gm in sub.groupby('mouse_id'):
            points=gm.groupby('variant_id')['z'].mean()
            x=[i for i,v in enumerate(VARIANTS) if v in points.index]
            yy=[np.tanh(points[v]) for v in VARIANTS if v in points.index]
            ax.plot(x,yy,'o-',alpha=.45,lw=1,label=mouse)
        ax.set_xticks(range(len(VARIANTS)),VARIANTS,rotation=25,fontsize=8)
        ax.set_ylim(ROI_YLIM)
        ax.set_title(f'{roi}: {len(sub.mouse_id.unique())} mice')
        ax.axhline(0,color='k',alpha=.25,lw=.6)
        ax.grid(axis='y',alpha=.12)
    for ax in axes[:,0]:ax.set_ylabel('Pearson r = tanh(mouse mean z)')
    fig.suptitle(f'{ROI_MASK_SET} • {ROI_COMPARISON_GROUP} • {ROI_KIND} • purely descriptive')
    plt.show()


## 8. Saved FC maps for one recording, on a common Pearson-r color scale

Find a recording stem shared by all selected variants (or use the explicit `MAP_STEM`). Load the saved mean Fisher-z maps and display `tanh(z)` on the same Pearson-r scale. The original plotting cell has a fixed 2 × 2 layout even if fewer than four variants are selected.


In [ ]:
MAP_MASK_SET = 'daypt_anesthesia'
MAP_STEM = None  # None selects the first stem with maps; otherwise provide an exact stem name.
MAP_KIND = 'homotopic'  # homotopic | gbc
MAP_VMIN, MAP_VMAX = -1.0, 1.0
MAP_CMAP = 'RdBu_r'

filename = 'homotopic_z_mean.npy' if MAP_KIND=='homotopic' else 'gbc_z_mean.npy'
choices=[]
for vid,d in runs.items():
    x=d['index']; x=x[x['mask_set'].eq(MAP_MASK_SET)]
    choices.append(set(x['stem'].astype(str)))
common_stems=set.intersection(*choices) if choices else set()
selected_stem = (MAP_STEM if MAP_STEM is not None else (sorted(common_stems)[0] if common_stems else None))
if selected_stem is None:
    print('Нет общей записи для всех выбранных вариантов.')
else:
    fig, axs=plt.subplots(2,2,figsize=(11,9),layout='constrained')
    im=None
    for ax, vid in zip(axs.ravel(),VARIANTS):
        if vid not in runs:
            ax.axis('off');continue
        row=runs[vid]['index']; row=row[(row['mask_set']==MAP_MASK_SET)&(row['stem']==selected_stem)]
        if len(row)!=1:
            ax.set_title(f'{vid}: запись недоступна');ax.axis('off');continue
        rr=row.iloc[0]; directory=s7._stem_output_dirs(runs[vid]['paths'],MAP_MASK_SET,rr)['stem_maps']
        if not (directory/filename).exists():
            ax.set_title(f'{vid}: карта отсутствует');ax.axis('off');continue
        zz=np.load(directory/filename)
        im=ax.imshow(np.tanh(zz),cmap=MAP_CMAP,vmin=MAP_VMIN,vmax=MAP_VMAX)
        ax.set_title(f'{vid} • {selected_stem}');ax.axis('off')
    if im is not None: fig.colorbar(im,ax=axs.ravel().tolist(),label='Pearson r',shrink=.7)
    plt.show()


## Outputs and subsequent publication analysis

Each directory `<out>/step7_functional_connectivity/<rest_variant>/<variant_id>/` stores the existing file names `homotopic_roi_values.csv`, `global_roi_values.csv`, `homotopic_z_mean.npy`, `gbc_z_mean.npy`, `fc_geometry.json`, `whole_cortex_fc_stem_values.csv`, and `whole_cortex_fc_mouse_values.csv`, when the corresponding computations and output options are enabled. A `variant_registry.csv` helps locate the selected variant runs.

**Compatibility note:** the historical `publication_analysis_v12_5_6.py` contains fixed references to an older Step7 structure and version checks. Migration to an explicitly selected `variant_id` must be handled separately. Do not combine results from distinct signal/filter variants into a single inferential analysis without an explicit method.
